<a href="https://colab.research.google.com/github/miso-20/ESSA/blob/main/ESAA_OB_WEEK_04_1-review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **수상작 리뷰**

문맥 기반 문장 순서 예측 AI 경진대회

https://dacon.io/competitions/official/236489/codeshare/12537?page=1&dtype=recent


## **주제**
뒤섞인 한국어 문장의 올바른 순서를 예측하는 AI 알고리즘 개발



## **데이터**

1. train.csv
- ID : 샘플별 고유 ID
- sentence_0, sentence_1, sentence_2, sentence_3: 섞여 있는 문장 4개
- answer_0, answer_1, answer_2, answer_3 : 올바른 문장 순서 배열 (각 값은 sentence_* 컬럼을 의미함)


2. test.csv
- ID : 샘플별 고유 ID
- sentence_0, sentence_1, sentence_2, sentence_3 : 섞여 있는 문장 4개


## **코드 흐름**

### 1. 데이터 로드 및 전처리
- 주어진 train_df의 문장들을 정답 순서에 맞춰 재배열함

- augment_data_fixed 함수를 사용하여 4개 문장의 배치 순서를 무작위로 섞어 데이터를 증강함

- 모델 학습을 위해 "문장을 순서대로 정렬하세요: [문장들]" 형태의 input과 순서를 공백으로 구분한 target 텍스트로 변환함

### 2. 모델 로드 및 파인튜닝 설정 (PEFT)

- unsloth 라이브러리의 FastLanguageModel을 활용해 Qwen/Qwen3-14B 모델을 4-bit 양자화 상태로 로드함

- 메모리 효율적인 학습을 위해 주요 모듈(q_proj, k_proj 등)에 LoRA(r=32, alpha=32)를 적용함

### 3. 모델 학습 및 앙상블 추론
- SFTTrainer를 통해 AdamW 8-bit 옵티마이저와 코사인 스케줄러를 적용하여 모델을 학습함

- predict_order 함수로 텍스트를 생성한 뒤, 정답 문자열을 파싱하여 정수 리스트로 반환함

-단일 모델 예측 후, 파인튜닝된 여러 모델의 결과를 앙상블(Hard voting)하여 최종 제출물을 생성함



**주요 코드**

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name= CFG['MODEL_NAME'],
    max_seq_length=4096,
    dtype=None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)

## **새롭게 알게 된 내용 / 어려운 내용 / 배울 점**

- LLM 파인튜닝 최적화: unsloth 라이브러리와 4-bit 양자화를 활용하면 제한된 자원(Colab A100 등)에서도 14B 크기의 대형 모델(Qwen3-14B)을 효율적으로 학습시킬 수 있다는 점을 알게 됨

- 데이터 증강 전략: 단순히 주어진 데이터 셋만 사용하는 것이 아니라, random.sample을 이용해 4개 문장의 순서 조합을 다양하게 섞어(augment_data_fixed) 학습 데이터를 직접 늘리는 접근법이 인상 깊음

- 안정적인 예외 처리: 생성형 AI 특성상 예측 출력물의 포맷이 어긋날 수 있는데, predict_order 함수 내에 try-except ValueError 구문을 넣어 파싱 오류 시 기본값 [0, 1, 2, 3]을 반환하도록 방어 코드를 작성한 점이 실무적으로 배울 점임